In [1]:
import torch
def print_gpu_info():
    """Prints CUDA GPU information if available."""
    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    gpu_id = torch.cuda.current_device()

    print(f"GPU : {torch.cuda.get_device_name(gpu_id)}")
    print(f"GPU capability: {torch.cuda.get_device_capability(gpu_id)}")

    gpu_props = torch.cuda.get_device_properties(gpu_id)

    print(f"GPU Memory : {gpu_props.total_memory / 1024**3:.2f} GB")
    print(f"Multiprocessors: {gpu_props.multi_processor_count}")
    print(f"Max Threads/MP : {gpu_props.max_threads_per_multi_processor}")
    
print_gpu_info()


GPU : NVIDIA GeForce RTX 3060
GPU capability: (8, 6)
GPU Memory : 11.63 GB
Multiprocessors: 28
Max Threads/MP : 1536


In [3]:

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import accuracy_score, classification_report

from transformers import AutoImageProcessor, AutoModel



In [4]:
checkpoint = "microsoft/resnet-50"

processor = AutoImageProcessor.from_pretrained(checkpoint)
backbone = AutoModel.from_pretrained(checkpoint)

backbone.eval()
backbone.requires_grad_(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone.to(device)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ResNetModel(
  (embedder): ResNetEmbeddings(
    (embedder): ResNetConvLayer(
      (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (encoder): ResNetEncoder(
    (stages): ModuleList(
      (0): ResNetStage(
        (layers): Sequential(
          (0): ResNetBottleNeckLayer(
            (shortcut): ResNetShortCut(
              (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (layer): Sequential(
              (0): ResNetConvLayer(
                (convolution): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalizatio

In [5]:
class HFImageDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, processor):
        self.dataset = datasets.ImageFolder(root_dir)
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values, label


In [8]:
BATCH_SIZE = 16
DATA_DIR = "handbags-shoes"
train_dataset = HFImageDataset(os.path.join(DATA_DIR, "train"), processor)
val_dataset   = HFImageDataset(os.path.join(DATA_DIR, "validation"), processor)
test_dataset  = HFImageDataset(os.path.join(DATA_DIR, "test"), processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.dataset.classes
print("Classes:", class_names)


Classes: ['handbags', 'shoes']


In [10]:
def extract_features(dataloader, model):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)

            # ResNet output: (batch, channels, H, W)
            feature_maps = outputs.last_hidden_state
            features = feature_maps.mean(dim=[2, 3])  # Global Average Pooling

            all_features.append(features.cpu())
            all_labels.append(labels)

    return torch.cat(all_features), torch.cat(all_labels)


In [11]:
train_features, train_labels = extract_features(train_loader, backbone)
val_features, val_labels     = extract_features(val_loader, backbone)
test_features, test_labels   = extract_features(test_loader, backbone)

print("Train features:", train_features.shape)


Train features: torch.Size([98, 2048])


In [12]:
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


In [13]:
model = Classifier(train_features.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20


In [14]:
for epoch in range(EPOCHS):
    model.train()

    optimizer.zero_grad()
    outputs = model(train_features.to(device))
    loss = criterion(outputs, train_labels.float().to(device))
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(val_features.to(device))
        val_preds = (torch.sigmoid(val_outputs) > 0.5).int()
        val_acc = accuracy_score(val_labels, val_preds.cpu())

    print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")


Epoch 01 | Loss: 0.6893 | Val Acc: 1.0000
Epoch 02 | Loss: 0.6229 | Val Acc: 1.0000
Epoch 03 | Loss: 0.5602 | Val Acc: 1.0000
Epoch 04 | Loss: 0.4969 | Val Acc: 1.0000
Epoch 05 | Loss: 0.4352 | Val Acc: 1.0000
Epoch 06 | Loss: 0.3713 | Val Acc: 1.0000
Epoch 07 | Loss: 0.3127 | Val Acc: 1.0000
Epoch 08 | Loss: 0.2653 | Val Acc: 1.0000
Epoch 09 | Loss: 0.2212 | Val Acc: 1.0000
Epoch 10 | Loss: 0.1866 | Val Acc: 1.0000
Epoch 11 | Loss: 0.1567 | Val Acc: 1.0000
Epoch 12 | Loss: 0.1257 | Val Acc: 1.0000
Epoch 13 | Loss: 0.0982 | Val Acc: 1.0000
Epoch 14 | Loss: 0.0835 | Val Acc: 1.0000
Epoch 15 | Loss: 0.0681 | Val Acc: 1.0000
Epoch 16 | Loss: 0.0547 | Val Acc: 1.0000
Epoch 17 | Loss: 0.0447 | Val Acc: 1.0000
Epoch 18 | Loss: 0.0368 | Val Acc: 1.0000
Epoch 19 | Loss: 0.0308 | Val Acc: 1.0000
Epoch 20 | Loss: 0.0238 | Val Acc: 1.0000


In [15]:
model.eval()
with torch.no_grad():
    test_outputs = model(test_features.to(device))
    test_preds = (torch.sigmoid(test_outputs) > 0.5).int()

test_acc = accuracy_score(test_labels, test_preds.cpu())
print("Test Accuracy:", test_acc)

print(classification_report(test_labels, test_preds.cpu(), target_names=class_names))


Test Accuracy: 1.0
              precision    recall  f1-score   support

    handbags       1.00      1.00      1.00        21
       shoes       1.00      1.00      1.00        18

    accuracy                           1.00        39
   macro avg       1.00      1.00      1.00        39
weighted avg       1.00      1.00      1.00        39



In [16]:
checkpoint = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(checkpoint)
vit_base = AutoModel.from_pretrained(checkpoint)

vit_base.eval()
vit_base.requires_grad_(False)
vit_base.to(device)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [17]:
class HFImageDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, processor):
        self.dataset = datasets.ImageFolder(root_dir)
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values, label
    
BATCH_SIZE = 16
DATA_DIR = "handbags-shoes"
train_dataset = HFImageDataset(os.path.join(DATA_DIR, "train"), processor)
val_dataset   = HFImageDataset(os.path.join(DATA_DIR, "validation"), processor)
test_dataset  = HFImageDataset(os.path.join(DATA_DIR, "test"), processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.dataset.classes
class_names


['handbags', 'shoes']

In [18]:
def extract_vit_features(dataloader, model):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)

            outputs = model(images)
            # outputs.last_hidden_state shape:
            # (batch_size, num_tokens, hidden_dim)

            cls_features = outputs.last_hidden_state[:, 0, :]  # CLS token
            all_features.append(cls_features.cpu())
            all_labels.append(labels)

    return torch.cat(all_features), torch.cat(all_labels)


In [19]:
hidden_dim = 768
train_features, train_labels = extract_vit_features(train_loader, vit_base)
val_features, val_labels     = extract_vit_features(val_loader, vit_base)
test_features, test_labels   = extract_vit_features(test_loader, vit_base)

train_features.shape


torch.Size([98, 768])

In [20]:
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)
model = Classifier(train_features.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
def accuracy_torch(logits, labels):
    preds = (torch.sigmoid(logits) > 0.5).long()
    correct = (preds == labels).sum().item()
    return correct / len(labels)


In [21]:
for epoch in range(EPOCHS):
    model.train()

    optimizer.zero_grad()
    outputs = model(train_features.to(device))
    loss = criterion(outputs, train_labels.float().to(device))
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(val_features.to(device))
        val_acc = accuracy_torch(val_outputs.cpu(), val_labels)

    print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")


Epoch 01 | Loss: 0.7348 | Val Acc: 0.9600
Epoch 02 | Loss: 0.3407 | Val Acc: 1.0000
Epoch 03 | Loss: 0.1394 | Val Acc: 1.0000
Epoch 04 | Loss: 0.0770 | Val Acc: 1.0000
Epoch 05 | Loss: 0.0402 | Val Acc: 1.0000
Epoch 06 | Loss: 0.0223 | Val Acc: 1.0000
Epoch 07 | Loss: 0.0137 | Val Acc: 1.0000
Epoch 08 | Loss: 0.0102 | Val Acc: 1.0000
Epoch 09 | Loss: 0.0066 | Val Acc: 1.0000
Epoch 10 | Loss: 0.0036 | Val Acc: 1.0000
Epoch 11 | Loss: 0.0022 | Val Acc: 1.0000
Epoch 12 | Loss: 0.0013 | Val Acc: 1.0000
Epoch 13 | Loss: 0.0015 | Val Acc: 1.0000
Epoch 14 | Loss: 0.0010 | Val Acc: 1.0000
Epoch 15 | Loss: 0.0009 | Val Acc: 1.0000
Epoch 16 | Loss: 0.0007 | Val Acc: 1.0000
Epoch 17 | Loss: 0.0004 | Val Acc: 1.0000
Epoch 18 | Loss: 0.0004 | Val Acc: 1.0000
Epoch 19 | Loss: 0.0003 | Val Acc: 1.0000
Epoch 20 | Loss: 0.0003 | Val Acc: 1.0000


In [22]:
model.eval()
with torch.no_grad():
    test_outputs = model(test_features.to(device))
    test_acc = accuracy_torch(test_outputs.cpu(), test_labels)

print("Test Accuracy:", test_acc)


Test Accuracy: 1.0


In [23]:
def confusion_matrix_binary(logits, labels):
    preds = (torch.sigmoid(logits) > 0.5).long()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    return tp, tn, fp, fn

tp, tn, fp, fn = confusion_matrix_binary(test_outputs.cpu(), test_labels)
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")


TP=18, TN=21, FP=0, FN=0
